<a href="https://colab.research.google.com/github/dashsumit/Automated_Text_Summerization-/blob/main/Automated__English_Hybrid(Extractive_%2B_Abstractive_)_Text_Summerizatio_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Extractive Text Summerization

In [ ]:
# # for extractive summerization  we will use sentence importance to generate the extractive summary of the input text
# 1. we will tokenize the text and identify the ocuurence of important words
#    then we will calculate the importance of each word .

# 2. next we will assign weights two sentences  based on on the sum of the important  score of there words .

# 3. finally we will rank the sentences and select top and sentences from the text body  to form the summary

In [ ]:
import spacy
from spacy.lang.en.stop_words import STOP_WORDS
from string import punctuation
from collections import Counter
import pandas as pd
from heapq import nlargest,nsmallest

In [ ]:
text = '''
At the heart of cutting-edge innovation is a silent revolution—artificial intelligence. It is not just a scientific field, but a dream come true: one in which machines can mimic, and even surpass, human intelligence. The main goal of artificial intelligence is to teach machines—to think, learn, and adapt—just as we do.

But how is this possible? The answer lies in the very heart of artificial intelligence, machine learning. Machine learning is a method by which machines learn from data. Just as a child learns to recognize faces, say words, or solve puzzles through experience, a machine learns from examples, patterns, and results.

Artificial intelligence is not limited to learning; it also extends to perceiving, reasoning, and acting. Machines can understand human language through natural language processing (NLP). They can analyze images and scenes through computer vision. They use neural networks to mimic the structure of the human brain—processing complex data through layers of “neurons.”

One of the most powerful architectures in machine learning is the deep learning model, where machines learn abstract features through layers of learning. It’s the technology that’s used to create voice assistants, smart translators, and even works of art.

But beyond algorithms or models, AI is a reflection of us—our creativity, our pursuit of reason, and our intelligence. With power comes responsibility, and ethical AI reminds us that we must build a future that is fair, transparent, and beneficial for all.

Machine learning and AI are not just technologies—they are a bridge between the human mind and the world of machines, with the potential to transform every aspect of life—from medicine to music, from education to exploration.
'''

In [ ]:
len(text)

Small Size englis model from spacy

In [ ]:
nlp=spacy.load('en_core_web_sm')
nlp

applying the loaded pretrain model

In [ ]:
doc = nlp(text)
doc

Code For tokenizations and removing stopwords from our text



In [ ]:
# using list comprehension

tokens= [token.text.lower() for token in doc
         if not token.is_stop and
         not token.is_punct and
         token.text != '\n']

In [ ]:
tokens

Alternative approch tokenizations and removing stopwords from our text using pos taging

In [ ]:
tokens1=[]
stopwords = list(STOP_WORDS)
allowed_pos = ['ADJ','PROPN','VERB','NOUN']
for token in doc :
  if token.text in stopwords or token.text in punctuation:
    continue
  if token.pos_ in allowed_pos:
    tokens1.append(token.text)

In [ ]:
tokens1

Frequency of the sentences

In [ ]:
# for extractive we have to find the sentences that have most  important word in them by focusing on sentences with the keywords appearing most often .ImportWarning

# we create the summary that capture the main idea of the summary .

In [ ]:
word_freq=Counter(tokens)
word_freq

In [ ]:
#pos approach
word_freq1=Counter(tokens1)
word_freq1

In [ ]:
max_freq= max(word_freq.values())
max_freq

In [ ]:
#pos approach
max_freq1= max(word_freq1.values())
max_freq1

Normalizing the values

In [ ]:
for word in word_freq.keys():
  word_freq[word] = word_freq[word] / max_freq

word_freq

In [ ]:
#pos approach

for word in word_freq1.keys():
  word_freq1[word] = word_freq1[word] / max_freq1

word_freq1

Sentence Tokenizations

In [ ]:
sent_token = [sent.text for sent in doc.sents]
sent_token

writing logic to calculate sentences score based on word frequencies

In [ ]:
sent_score= {}
for sent in sent_token:
  for word in sent.split():
    if word.lower() in word_freq.keys():
      if sent not in sent_score.keys():
        sent_score[sent] = word_freq[word.lower()]
      else:
        sent_score[sent] += word_freq[word]
    print(word)


In [ ]:
sent_score

Creating a dataframe in pandas

In [ ]:
df =pd.DataFrame(sent_score.items(),columns=['sentence','score'])
df

In [ ]:
#as we are doing extractive summarization , so we have to select top most scores

In [ ]:
number_of_sentences = 3
n= nlargest(number_of_sentences,sent_score,key =sent_score.get)
# key =sent_score.get
# use to extract the score from df to comapre the largest
n

In [ ]:
extractive_summerization = " " .join(n)
extractive_summerization

In [ ]:
len(extractive_summerization)

# End Of Extractive Text Summerrization

# Abstractive text Summerization

In [ ]:
from transformers import pipeline

In [ ]:
summerizer = pipeline("summarization", model= 't5-base', tokenizer='t5-base', framework='pt')

Applying in to the model

In [ ]:
summery = summerizer(text, max_length=50, min_length=30, do_sample=False)

In [ ]:
summery

printing summary into proper format

In [ ]:
print(summery[0]['summary_text']) # Changed 'summery_text' to 'summary_text'
# summary_text = summery[0]['summary_text']
# for line in summary_text.split('\n'):
#     print(line)

# summary_text = " ".join(summary_text.split())
# summary_text

In [ ]:
summary_text=summery[0]['summary_text']
summary_text

In [ ]:
len(summary_text)

# End Of Abstractive Summerization

# Hybrid(Extractive + Abstractive) Summerization

In [ ]:
# Hybrid summarization (simple concatenation)
hybrid_summary =  extractive_summerization + " " + summary_text
hybrid_summary

In [ ]:
len(hybrid_summary)